In [14]:
import os
import time
import torch
import torchinfo
import numpy as np
import pandas as pd

import scr.Logic as Logic
import scr.Models as Models
import scripts.Validation as Validation

In [12]:
MODEL_NAME = 'BasicTD_004'
MODEL = f'{MODEL_NAME}.pickle'
MODEL_TYPE = Models.Model_BasicTD
INPUT_SIZE = (1,198)

model = Models.Model_Loader.load_model(MODEL)

## Data

Our task does not include a traditional data set, and instead involves the use of an environment on which an agent learns. The species of how this agent learns and how this relates to traditional supervised methods learned in this class is described below. Here, however, is an explanation of how the environment is constructed and our use of the GNUBG engine. 

Our project uses a custom board class that represents a current position in backgammon. This, along with a variable to keep track of whose turn it is and a simple function to generate a random dice roll is all that is needed to play a game of backgammon. The class includes a variety of methods from helper functions such as `is_game_over` to `return_legal_moves` and `execute_move` that allow our agents to consider and then play moves turn after turn until the end of a game is reached. In the reinforcement learning spaces a single complete game is often considered an "episode" and can be thought of in a similar way to epochs. These terms are used interchangeably in this report. While a custom class, `Board` does use a third party library to generate the possible legal moves. This is done both for simplicity and to prevent unnecessary training bottlenecks. More details on this library can be found in the project readme. 

The board also has several methods that allow for the reference of the GNUBG engine. This engine is supported via another python library and is used both as our main method for evaluating the models and also for an experimental model in which we use GNUBG evaluations as the target prediction values at each training step. `return_gnubg_win_probs` will return the engine evaluation for each board state and is used throughout the project. We also use a GNUBG function to return the best move which is used to test the model accuracy by checking against the final board state generated by the move (as is backgammon multiple moves and lead to the same outcome). 

Unfortunately, our board class, GNUBG, and gym Backgammon all use different positional encodings. This is mainly solved by a variety of helper functions found in the board class that serve to convert or translate between the different encodings. Additionally, as part of our experiment was to use a variety of feature representations there are additional conversion functions to the Tesauro encoding and a custom encoding that we developed. As an example of this, the encoding method used by our class and gym Backgammon is shown below.

Finally, for one of our experimental models we decided to try to use a more traditional supervised learning approach to train a model. This involved having GNUBG play 10,000 games against itself and at each move record the position and its evaluation. This generated a csv file with over 600,000 observations. This positional encodings for this use the tesauro method. 

![Position Encodings](BG_Board_Positions.png)

Below you will find a data frame of the training data file used for our traditional supervised learning model. Additionally, we print a a backgammon board in it's starting configuration and list the possible legal moves for the first player rolling a 2,3. Moves are represented as a list of tuples where the first element is the starting position and the second element is the ending position. A move with multiple tuples represents a move with multiple pieces.

In [ ]:
# Data - 1

data = pd.read_csv('data/training_data.csv')
data

,state,win_prob
0,"[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.533846
1,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",0.495789
2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.534481
3,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.526004
4,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.596460
...,...,...
626661,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000000
626662,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000000
626663,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000000
626664,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000000


In [ ]:
# Data - 2

board = Logic.Board()
board.render_terminal()
moves = board.return_legal_moves(player=1, dice=(2,3))
print(*moves, sep='\n')

|    | 00   01   02   03   04   05 |    | 06   07   08   09   10   11 |    |
|----|----+----+----+----+----+----|----|----+----+----+----+----+----|----|
| 00 |  X                        O |    |       O                   X |    |
|    |  X                        O | P2 |       O                   X |    |
|    |                           O |    |       O                   X |    |
|    |                           O | O  |                           X |    |
|    |                           O | 00 |                           X |    |
|    |                             |    |                             |    |
|----|                             |    |                             |----|
|    |                             |    |                             |    |
|    |                           X | 00 |                           O |    |
|    |                           X | X  |                           O |    |
|    |                           X |    |       X                   O |    |